# Notebook 2 — Faces com OpenCV: Detecção Sequencial e Otimização Paralela

Este laboratório usa um subconjunto com **100 imagens de faces** organizadas em **subpastas** (uma subpasta por indivíduo), no estilo `VGGFaceHQ`.

## Objetivos
1. Executar um pipeline **sequencial** de detecção facial.
2. Gerar uma estrutura de resultados em memória contendo:
   - caminho da imagem
   - `bbox` da face
   - pontos aproximados de:
     - olho esquerdo
     - olho direito
     - nariz
     - canto esquerdo da boca
     - canto direito da boca
3. Visualizar as **5 primeiras detecções válidas**.
4. Medir o **tempo sequencial**.
5. Implementar duas otimizações:
   - **Fork / Divide-and-Conquer conceitual**
   - **Multiprocessing**
6. Calcular **speedup** e **eficiência**.

## Observação importante
Neste notebook, usaremos **OpenCV**. Para manter o ambiente simples e replicável, os “landmarks” serão obtidos por detecção clássica com Haar cascades:
- face
- olhos
- nariz
- boca

Eles são **aproximações úteis para estudo**, não um detector biométrico de produção.

## Estrutura esperada do dataset

A pasta raiz deve conter subpastas, por exemplo:

```text
faces/
├── pessoa_001/
│   ├── img_001.jpg
│   ├── img_002.jpg
├── pessoa_002/
│   ├── img_001.jpg
│   ├── img_002.jpg
...
```

Ajuste os caminhos na próxima célula conforme sua máquina.

In [ ]:
from pathlib import Path
import os
import time
import math
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# CONFIGURAÇÃO DE CAMINHOS
# ==========================================

# Ajuste para a pasta raiz com as 200 imagens de faces
DATASET_DIR = Path("faces")

# Extensões válidas
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert DATASET_DIR.exists(), f"Pasta não encontrada: {DATASET_DIR.resolve()}"
print("Dataset:", DATASET_DIR.resolve())

## 1. Encontrando todas as imagens do dataset

Como o dataset está organizado em subpastas, faremos uma busca recursiva.

In [ ]:
def list_image_files(root_dir: Path, valid_exts=VALID_EXTENSIONS):
    files = []
    for path in root_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in valid_exts:
            files.append(path)
    files = sorted(files)
    return files

image_files = list_image_files(DATASET_DIR)
print(f"Total de imagens encontradas: {len(image_files)}")
print("Exemplos:")
for p in image_files[:5]:
    print(" -", p)

## 2. Carregando os detectores OpenCV

Usaremos cascatas Haar distribuídas com o próprio OpenCV, quando disponíveis.

### Detectores
- Face: `haarcascade_frontalface_default.xml`
- Olhos: `haarcascade_eye.xml`
- Nariz: `haarcascade_mcs_nose.xml` ou fallback simples
- Boca: `haarcascade_smile.xml`

Nem todo ambiente traz todos os arquivos. O notebook já trata isso de forma segura.

In [ ]:
def load_cascade(filename: str):
    cascade_path = Path(cv2.data.haarcascades) / filename
    if cascade_path.exists():
        return cv2.CascadeClassifier(str(cascade_path))
    return None

FACE_CASCADE = load_cascade("haarcascade_frontalface_default.xml")
EYE_CASCADE = load_cascade("haarcascade_eye.xml")
NOSE_CASCADE = load_cascade("haarcascade_mcs_nose.xml")  # pode não existir em alguns ambientes
MOUTH_CASCADE = load_cascade("haarcascade_smile.xml")

assert FACE_CASCADE is not None, "Cascade de face não encontrada."

print("Face cascade carregada:", FACE_CASCADE is not None)
print("Eye cascade carregada:", EYE_CASCADE is not None)
print("Nose cascade carregada:", NOSE_CASCADE is not None)
print("Mouth cascade carregada:", MOUTH_CASCADE is not None)

## 3. Funções auxiliares

Vamos:
1. carregar a imagem
2. detectar a face principal
3. restringir as buscas de olhos, nariz e boca à região da face
4. converter as detecções em **pontos centrais**

In [ ]:
def rect_center(x, y, w, h):
    return (int(x + w / 2), int(y + h / 2))

def choose_largest_rect(rects):
    if len(rects) == 0:
        return None
    rects = sorted(rects, key=lambda r: r[2] * r[3], reverse=True)
    return rects[0]

def choose_two_eyes(eyes):
    if len(eyes) < 2:
        return None
    eyes = sorted(eyes, key=lambda e: e[0])  # esquerda -> direita
    left = eyes[0]
    right = eyes[-1]
    return left, right

def choose_nose(noses, face_h):
    if len(noses) == 0:
        return None
    # Preferir nariz na região central da face
    noses = sorted(noses, key=lambda n: abs((n[1] + n[3] / 2) - face_h * 0.5))
    return noses[0]

def choose_mouth(mouths, face_h):
    if len(mouths) == 0:
        return None
    # Preferir boca na parte inferior da face
    mouths = sorted(mouths, key=lambda m: -(m[1] + m[3] / 2))
    return mouths[0]

In [ ]:
def detect_face_and_landmarks_opencv(image_path: Path):
    '''
    Retorna um dicionário com:
    - image
    - bbox
    - landmarks (olhos, nariz, boca)
    ou None se nenhuma face for detectada.
    '''
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detecta faces
    faces = FACE_CASCADE.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(faces) == 0:
        return None

    x, y, w, h = choose_largest_rect(faces)
    face_roi_gray = gray[y:y+h, x:x+w]

    landmarks = {
        "left_eye": None,
        "right_eye": None,
        "nose": None,
        "mouth_left": None,
        "mouth_right": None,
    }

    # ------------------------------------------
    # Olhos
    # ------------------------------------------
    if EYE_CASCADE is not None:
        upper_face = face_roi_gray[: int(h * 0.55), :]
        eyes = EYE_CASCADE.detectMultiScale(
            upper_face,
            scaleFactor=1.1,
            minNeighbors=6,
            minSize=(12, 12)
        )
        if len(eyes) >= 2:
            chosen = choose_two_eyes(eyes)
            if chosen is not None:
                left_eye, right_eye = chosen
                lx, ly, lw, lh = left_eye
                rx, ry, rw, rh = right_eye
                landmarks["left_eye"] = (int(x + lx + lw // 2), int(y + ly + lh // 2))
                landmarks["right_eye"] = (int(x + rx + rw // 2), int(y + ry + rh // 2))

    # ------------------------------------------
    # Nariz
    # ------------------------------------------
    if NOSE_CASCADE is not None:
        mid_face = face_roi_gray[int(h * 0.2): int(h * 0.8), :]
        noses = NOSE_CASCADE.detectMultiScale(
            mid_face,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(15, 15)
        )
        if len(noses) > 0:
            nx, ny, nw, nh = choose_nose(noses, mid_face.shape[0])
            ny_global = int(y + int(h * 0.2) + ny + nh // 2)
            nx_global = int(x + nx + nw // 2)
            landmarks["nose"] = (nx_global, ny_global)

    # ------------------------------------------
    # Boca (smile cascade como aproximação)
    # ------------------------------------------
    if MOUTH_CASCADE is not None:
        lower_face = face_roi_gray[int(h * 0.45):, :]
        mouths = MOUTH_CASCADE.detectMultiScale(
            lower_face,
            scaleFactor=1.2,
            minNeighbors=20,
            minSize=(25, 15)
        )
        if len(mouths) > 0:
            mx, my, mw, mh = choose_mouth(mouths, lower_face.shape[0])
            mouth_y = int(y + int(h * 0.45) + my)
            landmarks["mouth_left"] = (int(x + mx), int(mouth_y + mh // 2))
            landmarks["mouth_right"] = (int(x + mx + mw), int(mouth_y + mh // 2))

    def py_point(p):
        if p is None:
            return None
        return [int(p[0]), int(p[1])]

    result = {
        "image": str(image_path),
        "bbox": [int(x), int(y), int(w), int(h)],
        "landmarks": {
            "left_eye": py_point(landmarks["left_eye"]),
            "right_eye": py_point(landmarks["right_eye"]),
            "nose": py_point(landmarks["nose"]),
            "mouth_left": py_point(landmarks["mouth_left"]),
            "mouth_right": py_point(landmarks["mouth_right"]),
        }
    }
    return result


## 4. Teste rápido em uma única imagem

In [ ]:
if len(image_files) > 0:
    sample_result = detect_face_and_landmarks_opencv(image_files[0])
    print(json.dumps(sample_result, indent=2, ensure_ascii=False))
else:
    print("Nenhuma imagem encontrada.")


## 5. Função de visualização

Vamos desenhar:
- bounding box da face
- pontos dos olhos
- nariz
- canto esquerdo e direito da boca

In [ ]:
def draw_detection(result, figsize=(5, 5)):
    img = cv2.imread(result["image"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x, y, w, h = result["bbox"]
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

    for name, point in result["landmarks"].items():
        if point is not None:
            px, py = point
            cv2.circle(img, (px, py), 4, (255, 0, 0), -1)
            cv2.putText(
                img,
                name,
                (px + 4, py - 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255, 255, 0),
                1,
                cv2.LINE_AA
            )

    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(Path(result["image"]).name)
    plt.axis("off")
    plt.show()

## 6. Pipeline sequencial completo

Agora vamos processar todas as imagens **sequencialmente** e guardar os resultados em memória.

### Saída esperada
Uma lista de dicionários, um por imagem com face detectada.

In [ ]:
def run_sequential_face_pipeline(files):
    results = []
    start = time.perf_counter()

    for path in files:
        item = detect_face_and_landmarks_opencv(path)
        if item is not None:
            results.append(item)

    elapsed = time.perf_counter() - start
    return results, elapsed

sequential_results, T_seq = run_sequential_face_pipeline(image_files)

print(f"Detecções válidas: {len(sequential_results)}")
print(f"Tempo sequencial: {T_seq:.4f} s")

## 7. Visualizando as 5 primeiras detecções válidas

In [ ]:
preview = sequential_results[:5]
for item in preview:
    draw_detection(item, figsize=(4, 4))

## 8. Estruturando resultados em tabela

In [ ]:
def flatten_result(result):
    row = {
        "image": result["image"],
        "bbox_x": result["bbox"][0],
        "bbox_y": result["bbox"][1],
        "bbox_w": result["bbox"][2],
        "bbox_h": result["bbox"][3],
    }
    for k, v in result["landmarks"].items():
        if v is None:
            row[f"{k}_x"] = None
            row[f"{k}_y"] = None
        else:
            row[f"{k}_x"] = v[0]
            row[f"{k}_y"] = v[1]
    return row

df_seq = pd.DataFrame([flatten_result(r) for r in sequential_results])
df_seq.head()

## 9. Enunciado — Otimização 1: Fork / Divide-and-Conquer

### Tarefa
Implemente uma versão **conceitual** baseada em divisão recursiva da lista de arquivos.

### Ideia
- Se a lista for pequena, processe sequencialmente.
- Caso contrário:
  - divida a lista em duas metades
  - processe cada metade
  - combine os resultados

> Nesta etapa, o objetivo é exercitar o modelo **Fork–Join conceitual**.  
> Você pode implementar apenas a divisão e combinação, mesmo sem paralelismo real.

In [ ]:
# TODO: IMPLEMENTE AQUI SUA VERSÃO FORK / DIVIDE-AND-CONQUER

def run_fork_join_conceptual(files, threshold=20):
    '''
    Sugestão de estratégia:
    - caso base: lista pequena -> processa sequencialmente
    - caso recursivo: divide em duas metades e combina os resultados
    '''
    raise NotImplementedError("Implemente a versão Fork / Divide-and-Conquer.")

# Exemplo de uso:
# fork_results, T_fork = run_fork_join_conceptual(image_files, threshold=20)

## 10. Enunciado — Otimização 2: Multiprocessing

### Tarefa
Implemente uma versão paralela usando:
- `multiprocessing.Pool`, **ou**
- `concurrent.futures.ProcessPoolExecutor`

### Requisitos
- processar as imagens em paralelo
- retornar uma lista de detecções válidas
- medir o tempo total
- testar com diferentes números de processos: `p = 2, 4, 8`

### Dica
Cada worker pode:
1. receber o caminho de uma imagem
2. executar `detect_face_and_landmarks_opencv`
3. retornar `None` ou um dicionário válido

In [ ]:
# TODO: IMPLEMENTE AQUI SUA VERSÃO MULTIPROCESSING

def run_multiprocessing_face_pipeline(files, num_workers=4):
    '''
    Retorne:
    - results: lista de detecções válidas
    - elapsed: tempo total
    '''
    raise NotImplementedError("Implemente a versão multiprocessing.")

# Exemplo de uso:
# mp_results_4, T_mp_4 = run_multiprocessing_face_pipeline(image_files, num_workers=4)

## 11. Cálculo de métricas

Quando você terminar as implementações acima, use esta seção para calcular:

- **Speedup**: `S(p) = T_seq / T_par`
- **Eficiência**: `E(p) = S(p) / p`

In [ ]:
def compute_speedup(T_seq, T_par):
    return T_seq / T_par

def compute_efficiency(speedup, p):
    return speedup / p

In [ ]:
# Preencha estes valores após implementar as versões
# Exemplo:
# T_fork = ...
# T_mp_2 = ...
# T_mp_4 = ...
# T_mp_8 = ...

results_table = []

# if 'T_fork' in globals():
#     S_fork = compute_speedup(T_seq, T_fork)
#     results_table.append({
#         "config": "Fork conceitual",
#         "p": 1,
#         "tempo_s": T_fork,
#         "speedup": S_fork,
#         "eficiencia": S_fork
#     })

# if 'T_mp_2' in globals():
#     S2 = compute_speedup(T_seq, T_mp_2)
#     results_table.append({
#         "config": "Multiprocessing",
#         "p": 2,
#         "tempo_s": T_mp_2,
#         "speedup": S2,
#         "eficiencia": compute_efficiency(S2, 2)
#     })

# if 'T_mp_4' in globals():
#     S4 = compute_speedup(T_seq, T_mp_4)
#     results_table.append({
#         "config": "Multiprocessing",
#         "p": 4,
#         "tempo_s": T_mp_4,
#         "speedup": S4,
#         "eficiencia": compute_efficiency(S4, 4)
#     })

# if 'T_mp_8' in globals():
#     S8 = compute_speedup(T_seq, T_mp_8)
#     results_table.append({
#         "config": "Multiprocessing",
#         "p": 8,
#         "tempo_s": T_mp_8,
#         "speedup": S8,
#         "eficiencia": compute_efficiency(S8, 8)
#     })

df_metrics = pd.DataFrame(results_table)
df_metrics

## 12. Gráficos

Depois de preencher a tabela, gere gráficos para comparar:
- tempo
- speedup
- eficiência

In [ ]:
if not df_metrics.empty:
    plt.figure(figsize=(8, 4))
    plt.bar(df_metrics["config"] + " p=" + df_metrics["p"].astype(str), df_metrics["tempo_s"])
    plt.ylabel("Tempo (s)")
    plt.title("Tempo por configuração")
    plt.xticks(rotation=20)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.bar(df_metrics["config"] + " p=" + df_metrics["p"].astype(str), df_metrics["speedup"])
    plt.ylabel("Speedup")
    plt.title("Speedup por configuração")
    plt.xticks(rotation=20)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.bar(df_metrics["config"] + " p=" + df_metrics["p"].astype(str), df_metrics["eficiencia"])
    plt.ylabel("Eficiência")
    plt.title("Eficiência por configuração")
    plt.xticks(rotation=20)
    plt.show()
else:
    print("Preencha a seção de métricas após implementar as versões paralelas.")

## 13. Perguntas para análise

Responda com base nos seus experimentos:

1. O processamento de faces escalou melhor que o esperado ou pior que o esperado?
2. O ganho foi linear? Se não, por quê?
3. O gargalo parece estar mais em:
   - leitura da imagem
   - detecção da face
   - detecção dos landmarks aproximados
   - serialização/gerenciamento dos processos
4. A versão multiprocessing apresentou saturação?
5. Qual configuração teve melhor eficiência?
6. Como esta prática se conecta com:
   - speedup
   - eficiência
   - overhead
   - Lei de Amdahl

## 14. Extensões opcionais

Se quiser ir além:

- variar o número de imagens: 50, 100, 200
- variar o número de processos: 1, 2, 4, 8
- medir o impacto de:
  - imagens maiores
  - imagens menores
  - maior número de subpastas
- salvar os resultados em CSV
- comparar Haar cascade com outro detector OpenCV